# Islamabad (1 kW)

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Locate raw files

In [ ]:
import os
import pandas as pd

TARGET_FILES = ['weather_data.csv', 'solar_generation_1kW.csv', 'Building_1.csv']
found_paths = {}

for root, dirs, files in os.walk(str(RAW_DIR)):
    for file in files:
        if file in TARGET_FILES:
            found_paths[file] = os.path.join(root, file)

## Build unified hourly dataset

In [ ]:
import requests
import numpy as np
import pvlib

LATITUDE, LONGITUDE = 33.6844, 73.0479
TILT, SURFACE_AZIMUTH = 33.7, 180.0

wind_url = "https://archive-api.open-meteo.com/v1/archive"
wind_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": "2024-01-01",
    "end_date": "2024-12-31",
    "hourly": "wind_speed_10m",
    "timezone": "Asia/Karachi"
}
wind_response = requests.get(wind_url, params=wind_params)
wind_data = wind_response.json()

wind_speed_kmh = np.array(wind_data['hourly']['wind_speed_10m'][:8760])
wind_speed_ms = wind_speed_kmh / 3.6

path_solar = found_paths['solar_generation_1kW.csv']
path_weather = found_paths['weather_data.csv']
path_bldg = found_paths['Building_1.csv']

df_solar = pd.read_csv(path_solar)
df_weather = pd.read_csv(path_weather)
df_bldg = pd.read_csv(path_bldg)

df_dataset = pd.DataFrame()
df_dataset['utctime'] = pd.date_range('2024-01-01 00:00:00', periods=8760, freq='h')
df_dataset['Power_Actual_W'] = df_solar['Hourly Data: AC inverter power (W)']
df_dataset['Power_PU'] = df_dataset['Power_Actual_W'] / 1000.0

df_dataset['Temp_Ambient'] = df_weather['Outdoor Drybulb Temperature [C]']
df_dataset['Humidity'] = df_weather['Outdoor Relative Humidity [%]']
df_dataset['Wind_Speed'] = wind_speed_ms

hours = df_bldg['Hour'] - 1
months = df_bldg['Month']
df_dataset['HOUR_SIN'] = np.sin(2 * np.pi * hours / 24.0)
df_dataset['HOUR_COS'] = np.cos(2 * np.pi * hours / 24.0)
df_dataset['MONTH_SIN'] = np.sin(2 * np.pi * months / 12.0)
df_dataset['MONTH_COS'] = np.cos(2 * np.pi * months / 12.0)

times = pd.date_range('2024-01-01 00:00:00', periods=8760, freq='h', tz='Asia/Karachi')
solpos = pvlib.solarposition.get_solarposition(times, LATITUDE, LONGITUDE)

dni = df_weather['Direct Solar Radiation [W/m2]'].values
dhi = df_weather['Diffuse Solar Radiation [W/m2]'].values
ghi = np.maximum(dhi + dni * pvlib.tools.cosd(solpos['zenith'].values), 0)

dni_extra = pvlib.irradiance.get_extra_radiation(times)
poa_components = pvlib.irradiance.get_total_irradiance(
    surface_tilt=TILT,
    surface_azimuth=SURFACE_AZIMUTH,
    solar_zenith=solpos['zenith'].values,
    solar_azimuth=solpos['azimuth'].values,
    dni=dni,
    ghi=ghi,
    dhi=dhi,
    dni_extra=dni_extra,
    model='haydavies'
)

df_dataset['Irradiance_POA'] = poa_components['poa_global'].values
df_dataset['GHI'] = ghi
df_dataset['Latitude'] = LATITUDE
df_dataset['Longitude'] = LONGITUDE
df_dataset['Tilt'] = TILT

FINAL_COLUMNS = [
    'utctime', 'Power_Actual_W', 'Power_PU',
    'Irradiance_POA', 'GHI', 'Temp_Ambient', 'Wind_Speed', 'Humidity',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'Latitude', 'Longitude', 'Tilt'
]
df_dataset = df_dataset[FINAL_COLUMNS]

## Save

In [ ]:
SAVE_PATH = OUT_DIR / 'Islamabad.csv'
df_dataset.to_csv(SAVE_PATH, index=False)